# Stock Sentiment Classifier (GRU)

Same pipeline as `IMDB_GRU.ipynb`, adapted to `stock_data.csv` (`Text`, `Sentiment` columns, labels `1` / `-1`).

In [ ]:
import re
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("TensorFlow version:", tf.__version__)


## 1. Load the dataset

In [ ]:
df = pd.read_csv("stock_data.csv")
print("Shape:", df.shape)
df.head(3)


## 2. Drop duplicates

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.duplicated().sum()

## 3. Clean the text

Same cleaning function as the IMDB notebooks: lowercase, strip HTML/tags, keep only letters, collapse whitespace.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)        # remove HTML tags e.g. <br />
    text = re.sub(r"[^a-z\s]", " ", text)     # keep only letters
    text = re.sub(r"\s+", " ", text).strip()  # collapse whitespace
    return text


In [ ]:
df["clean_text"] = df["Text"].apply(clean_text)

## 4. Map labels

Dataset uses `1` (positive) / `-1` (negative). Map to `1` / `0` for binary cross-entropy.

In [ ]:
df["label"] = df["Sentiment"].map({1: 1, -1: 0})

## 5. Train/test split

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["clean_text"], df["label"],
    test_size=0.2, random_state=42, stratify=df["label"]
)


## 6. Prepare Sequences for the Neural Networks

### Tokenization
Convert each cleaned tweet into a sequence of integers, where each integer represents a word's rank in the vocabulary (fit **only** on the training set).

### Padding
Neural networks need fixed-length input. We pad/truncate every sequence to `max_len=200`.

In [ ]:
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=200, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=200, padding="post", truncating="post")

y_train_arr = y_train.values
y_test_arr = y_test.values


## 7. Build and train the GRU model

In [ ]:
gru_model = Sequential([
    Embedding(input_dim=10000, output_dim=64, mask_zero=True),

    GRU(64),

    Dropout(0.3),

    Dense(32, activation="relu"),

    Dense(1, activation="sigmoid")
])

gru_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

history_gru = gru_model.fit(
    X_train_pad,
    y_train_arr,
    validation_split=0.1,
    epochs=20,
    batch_size=32,
    callbacks=[early_stop]
)


## 8. Evaluate

In [ ]:
y_pred_gru = (gru_model.predict(X_test_pad) > 0.5).astype(int)

print(classification_report(y_test_arr, y_pred_gru))


In [ ]:
accuracy_score(y_test_arr, y_pred_gru)

In [ ]:
cm = confusion_matrix(y_test_arr, y_pred_gru)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Negative", "Positive"])
disp.plot(cmap="Blues")
plt.title("Stock Sentiment - GRU Confusion Matrix")
plt.show()


## 9. Save the model and tokenizer

In [ ]:
# Save the trained GRU model
gru_model.save("stock_gru_sentiment_model.h5")

# Save the fitted tokenizer
with open("stock_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)
